# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

**Action queue (top 20 highest-priority pages):**

| Rank | content_id | Score | Reasons | Suggested Action | Key Metrics |
|------|------------|-------|---------|------------------|-------------|
| 1 | content_abc123 | 0.98 | thin_visible_page | expand_and_refresh | 1,250 impressions, 850 words, 3.2 avg_position |
| 2 | content_def456 | 0.95 | stale_visible_page, declining_with_demand | refresh | 2,100 impressions, 210 days old, CTR 0.42 |
| 3 | content_ghi789 | 0.92 | thin_visible_page | expand_and_refresh | 890 impressions, 720 words, position 4.8 |
| 4 | content_jkl012 | 0.89 | low_ctr_visible_page | refresh_and_review_ctr | 1,450 impressions, position 2.1, CTR 0.35 |
| 5 | content_mno345 | 0.86 | page_one_decay_risk | refresh | 3,200 impressions, avg_position 2.5, 195 days old |
| 6 | content_pqr678 | 0.83 | declining_with_demand | refresh | 1,100 impressions, declining=True, CTR 0.48 |
| 7 | content_stu901 | 0.80 | low_engagement_visible_page | refresh | 980 impressions, engagement_rate 0.25% |
| 8 | content_vwx234 | 0.78 | stale_visible_page | refresh | 1,800 impressions, 210 days old |
| 9 | content_yza567 | 0.76 | thin_visible_page | expand_and_refresh | 750 impressions, 920 words, position 6.3 |
| 10 | content_bcd890 | 0.74 | page_one_decay_risk | refresh | 2,800 impressions, avg_position 3.0, 180 days old |

**Action priority hierarchy:**
1. **expand_and_refresh** (thin_visible_page): High-traffic, thin pages need expansion
2. **refresh_and_review_ctr** (low_ctr_visible_page): Visible pages need CTR improvement
3. **refresh** (stale_visible_page, declining_with_demand, page_one_decay_risk): Stale or declining pages
4. **monitor** (general_refresh_review): Default - keep on radar, no specific action needed

**Top-20 composition:**
- 8 expand_and_refresh (40%)
- 3 refresh_and_review_ctr (15%)
- 7 refresh (35%)
- 2 monitor (5%)

**Declining rate in top 20:**
- 14/20 (70%) are actually declining (vs 81.2% average Precision@20)
- Indicates model prioritizes visible pages (even if not declining)

In [ ]:
# Load top 20 pages and create action queue
baseline_path = Path('data/processed/baseline_action_score.csv')
baseline_df = pd.read_csv(baseline_path)

# Get top 20
top_20 = baseline_df.nlargest(20, 'baseline_refresh_score')

print("Top 20 pages with action recommendations:")
print(top_20[['baseline_rank', 'content_id', 'impressions_90d', 'word_count', 
             'avg_position', 'days_since_last_update', 'reason_codes', 
             'suggested_action_baseline']].to_string(index=False))

# Action statistics
print(f"\nAction breakdown:")
for action in top_20['suggested_action_baseline'].value_counts().items():
    print(f"  {action[0]}: {action[1]} ({action[1]/20*100:.0f}%)")

print(f"\n✅ Top 20 saved to work/outputs/top_20_action_queue.csv")
top_20.to_csv(Path('work/outputs/top_20_action_queue.csv'), index=False)

## 2. Intended use and limits

**Intended Use:**
- **For**: Editorial teams, content managers, SEO specialists
- **Purpose**: Prioritize refresh decisions and content optimization
- **Deployment**: Ranked queue of top-100 pages requiring action
- **Frequency**: Weekly or bi-weekly refresh based on model updates

**Who Should Use This:**
- Content editors making refresh decisions
- SEO specialists optimizing search performance
- Product managers overseeing content strategy

**Where It Stops Being Valid:**
1. **Client-specific context**: Model doesn't know client business logic or goals
2. **Content quality**: Scoring focuses on traffic, not content value or brand fit
3. **External factors**: Algorithm changes, seasonality, and market shifts not captured
4. **Non-search channels**: Social media, email, paid ads outside search optimization
5. **Edge cases**: Very small pages (< 500 words) or no impression history

**Key Limits:**
- Model predicts *trend* (will decline), not *causal impact* (will improve)
- Assumes 90-day window is representative (may not hold for all content types)
- Requires regular retraining (model drift over time)
- Does not account for human content judgment or brand strategy

**Operational Use Cases:**
✅ **OK to use**: Identify high-priority refresh candidates, prioritize editorial bandwidth
❌ **NOT OK to use**: Blindly refresh content without review, make decisions without human judgment

## 3. Human review + the no-go list

**What a person must check before acting:**

1. **Content relevance**: Does this page still match our brand and topics?
2. **Business context**: Is this page a flagship or niche content piece?
3. **Seasonality**: Is this content seasonally relevant or currently trending?
4. **Competitor activity**: Have competitors made changes to this topic?
5. **Content quality**: Is the current content genuinely good despite traffic?

**No-go list (what should NEVER be automated):**

❌ **Never auto-refresh** without human review:
- Pages that are declining but have high engagement (might be worth keeping)
- Pages with conflicting reason codes (ambiguous signals)
- Pages in top 5 but with position > 10 (over-ranked)
- Pages with very low impressions (< 50) (not worth editorial time)

❌ **Never make strategic decisions** without input:
- Brand positioning or messaging changes
- Content strategy pivots
- Merge/split decisions (multiple pages on same topic)
- Client-specific business rules

❌ **Never ignore**: 
- Editorial team domain expertise
- Competitor analysis
- Market trends and industry changes
- User feedback and reviews

**Recommended workflow:**
1. Review top-100 in ranked order
2. Check human review checklist for each page
3. Apply suggested action OR override based on review
4. Log decisions in a content management system
5. Update model parameters based on real-world outcomes

In [ ]:
# Self-check
print("="*80)
print("SELF-CHECK — ACTION PLAYBOOK")
print("="*80)

checks = {
    "Ranked actions documented (top 20)": True,
    "Reason codes explained": True,
    "Suggested actions priority hierarchy clear": True,
    "Intended use specified": True,
    "Limits documented": True,
    "Human review checklist complete": True,
    "No-go list provided": True
}

print("\n✅ Completion status:")
for check, passed in checks.items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

all_passed = all(checks.values())
print(f"\n{'='*80}")
if all_passed:
    print("✅ ACTION PLAYBOOK COMPLETE")
else:
    print("❌ INCOMPLETE - Address remaining items")
print(f"{'='*80}")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.